In [ ]:
import os
from google.cloud import bigquery

project_id = os.environ.get("GCP_PROJECT", "cancer-multiomics")
client = bigquery.Client(project=project_id)

# When running locally, install compatibility shims for BQ Studio imports
try:
    from google.colab.sql import bigquery as _test
except ImportError:
    import bq_compat
    bq_compat.install(client)

# Cancer Multi-Omics EDA

This notebook explores publicly available cancer genomics, transcriptomics, and proteomics data hosted on [BigQuery by ISB-CGC](https://isb-cgc.appspot.com/). The goal is to understand the shape, scale, and join-ability of these datasets before building any multi-omic analysis pipelines.

**Data sources:**
- **TCGA** (The Cancer Genome Atlas) — somatic mutations, RNA-seq expression, clinical data across 33 cancer types
- **CPTAC** (Clinical Proteomic Tumor Analysis Consortium) — mass-spec proteomics and phosphoproteomics

## Part 1: TCGA — Mutations and gene expression

### What columns does the mutation table have?

Before writing any analysis queries, we need to understand the schema. The `INFORMATION_SCHEMA.COLUMNS` view gives us every column name in the TCGA somatic mutation table. Key things to look for: patient identifiers, gene symbols, mutation coordinates, and variant classification.

In [2]:
# sql_engine: bigquery
# output_variable: mutation_column_names
# start _sql
_sql = """
SELECT column_name
FROM `isb-cgc-bq.TCGA.INFORMATION_SCHEMA.COLUMNS`
WHERE table_name = 'masked_somatic_mutation_hg38_gdc_current'
ORDER BY ordinal_position
""" # end _sql
from google.colab.sql import bigquery as _bqsqlcell
mutation_column_names = _bqsqlcell.run(_sql)
mutation_column_names

,column_name
0,project_short_name
1,case_barcode
2,primary_site
3,Hugo_Symbol
4,Entrez_Gene_Id
5,Center
6,NCBI_Build
7,Chromosome
8,Start_Position
9,End_Position


### How many patients per cancer type (clinical data)?

Querying the clinical table gives us the number of patients per TCGA project. This is our denominator — it tells us the total cohort size for each cancer type, regardless of which assays were performed.

In [3]:
# sql_engine: bigquery
# output_variable: cases_by_cancer_type_TCGA
# start _sql
_sql = """
SELECT proj__name, proj__project_id, COUNT(*) AS patient_count
FROM `isb-cgc-bq.TCGA.clinical_gdc_current`
GROUP BY proj__name, proj__project_id
ORDER BY patient_count DESC
""" # end _sql
from google.colab.sql import bigquery as _bqsqlcell
cases_by_cancer_type_TCGA = _bqsqlcell.run(_sql)
cases_by_cancer_type_TCGA

,proj__name,proj__project_id,patient_count
0,Breast Invasive Carcinoma,TCGA-BRCA,1098
1,Glioblastoma Multiforme,TCGA-GBM,617
2,Ovarian Serous Cystadenocarcinoma,TCGA-OV,608
3,Lung Adenocarcinoma,TCGA-LUAD,585
4,Uterine Corpus Endometrial Carcinoma,TCGA-UCEC,560
5,Kidney Renal Clear Cell Carcinoma,TCGA-KIRC,537
6,Head and Neck Squamous Cell Carcinoma,TCGA-HNSC,528
7,Brain Lower Grade Glioma,TCGA-LGG,516
8,Thyroid Carcinoma,TCGA-THCA,507
9,Lung Squamous Cell Carcinoma,TCGA-LUSC,504


### How many mutations per cancer type?

Now we look at the mutation table directly. Counting total rows and distinct patients per `project_short_name` tells us two things: how many patients have mutation data (not all clinical patients do), and the raw mutation counts per cancer type.

In [4]:
# sql_engine: bigquery
# output_variable: mutations_and_cases_by_cancer_type
# start _sql
_sql = """
SELECT project_short_name, COUNT(*) AS row_count, COUNT(DISTINCT case_barcode) AS patient_count
FROM `isb-cgc-bq.TCGA.masked_somatic_mutation_hg38_gdc_current`
GROUP BY project_short_name
ORDER BY patient_count DESC
""" # end _sql
from google.colab.sql import bigquery as _bqsqlcell
mutations_and_cases_by_cancer_type = _bqsqlcell.run(_sql)
mutations_and_cases_by_cancer_type

,project_short_name,row_count,patient_count
0,TCGA-BRCA,89568,968
1,TCGA-LUAD,194729,557
2,TCGA-UCEC,626945,511
3,TCGA-LGG,32780,509
4,TCGA-HNSC,87967,508
5,TCGA-PRAD,24779,490
6,TCGA-THCA,5834,488
7,TCGA-LUSC,172826,485
8,TCGA-SKCM,353450,467
9,TCGA-STAD,183107,431


### Mutation burden varies dramatically across cancer types

Raw mutation counts are misleading because cohort sizes differ. Dividing total mutations by patient count gives the average mutation burden per patient. This is a critical biological variable — hypermutated cancers like melanoma (UV damage) and endometrial cancer (MSI-high) have 10–20x more mutations than thyroid or prostate cancers.

In [5]:
mutations_and_cases_by_cancer_type['avg_mut_per_patient'] = mutations_and_cases_by_cancer_type['row_count'] / mutations_and_cases_by_cancer_type['patient_count']
mutations_and_cases_by_cancer_type

,project_short_name,row_count,patient_count,avg_mut_per_patient
0,TCGA-BRCA,89568,968,92.528926
1,TCGA-LUAD,194729,557,349.603232
2,TCGA-UCEC,626945,511,1226.898239
3,TCGA-LGG,32780,509,64.400786
4,TCGA-HNSC,87967,508,173.163386
5,TCGA-PRAD,24779,490,50.569388
6,TCGA-THCA,5834,488,11.954918
7,TCGA-LUSC,172826,485,356.342268
8,TCGA-SKCM,353450,467,756.852248
9,TCGA-STAD,183107,431,424.842227


### RNA-seq table schema

Shifting from mutations to gene expression. The RNA-seq table has a fundamentally different structure — let's see what columns are available. We're especially interested in the quantification columns (TPM, FPKM, raw counts) and the sample identifiers.

In [6]:
# sql_engine: bigquery
# output_variable: RNA_seq_column_names
# start _sql
_sql = """
SELECT column_name
FROM `isb-cgc-bq.TCGA.INFORMATION_SCHEMA.COLUMNS`
WHERE table_name = 'RNAseq_hg38_gdc_current'
ORDER BY ordinal_position
""" # end _sql
from google.colab.sql import bigquery as _bqsqlcell
RNA_seq_column_names = _bqsqlcell.run(_sql)
RNA_seq_column_names

,column_name
0,project_short_name
1,primary_site
2,case_barcode
3,sample_barcode
4,aliquot_barcode
5,gene_name
6,gene_type
7,Ensembl_gene_id
8,Ensembl_gene_id_v
9,unstranded


In [7]:
RNA_seq_column_names.to_string()

'           column_name\n0   project_short_name\n1         primary_site\n2         case_barcode\n3       sample_barcode\n4      aliquot_barcode\n5            gene_name\n6            gene_type\n7      Ensembl_gene_id\n8    Ensembl_gene_id_v\n9           unstranded\n10      stranded_first\n11     stranded_second\n12      tpm_unstranded\n13     fpkm_unstranded\n14  fpkm_uq_unstranded\n15    sample_type_name\n16         case_gdc_id\n17       sample_gdc_id\n18      aliquot_gdc_id\n19         file_gdc_id\n20            platform'

Barcode hierarchy:

*   case_barcode: AKA patient. Can have multiple samples taken.
*   sample_barcode: AKA tumor and matched normal tissue. Can have multiple aliquots.
*   aliquot_barcode: An aliquot is a physical portion sent to differnt labs.





### RNA-seq coverage by cancer type

How many patients have RNA-seq data per cancer type, and how large is the table? This helps us understand which TCGA projects have sufficient expression data for multi-omic analysis, and gives a sense of the BigQuery costs (expression tables are much larger than mutation tables due to their dense structure).

In [8]:
# sql_engine: bigquery
# output_variable: seq_and_patient_count_by_cancer_type
# start _sql
_sql = """
SELECT
  project_short_name,
  COUNT(*) AS row_count,
  COUNT(DISTINCT case_barcode) AS patient_count
FROM `isb-cgc-bq.TCGA.RNAseq_hg38_gdc_current`
GROUP BY project_short_name
ORDER BY patient_count DESC
""" # end _sql
from google.colab.sql import bigquery as _bqsqlcell
seq_and_patient_count_by_cancer_type = _bqsqlcell.run(_sql)
seq_and_patient_count_by_cancer_type

,project_short_name,row_count,patient_count
0,TCGA-BRCA,74677384,1095
1,TCGA-UCEC,35731096,557
2,TCGA-KIRC,37247696,533
3,TCGA-HNSC,34335824,521
4,TCGA-LUAD,36398400,517
5,TCGA-LGG,32394576,516
6,TCGA-THCA,34699808,505
7,TCGA-LUSC,34093168,501
8,TCGA-PRAD,33607856,497
9,TCGA-SKCM,28694072,469


In [9]:
# dataframe: seq_and_patient_count_by_cancer_type
# uuid: CC1849E3-13FF-4391-9870-F8F0EA5D4D5B
# output_variable:
# config_str: CuAZeyJjaGFydENvbmZpZyI6eyJkYXRhc291cmNlSWQiOiJfX1ZJWl9EQVRBU09VUkNFX18iLCJwcm9wZXJ0eUNvbmZpZyI6eyJjb21wb25lbnRQcm9wZXJ0eSI6eyJzb3J0IjpbeyJzb3J0RGlyIjoxLCJzb3J0Q29sdW1uIjoicXRfd3M3Znc3N2QxZCJ9XSwiYnJlYWtkb3duQ29uZmlnIjpbXSwiZmlsdGVycyI6W10sImluaGVyaXRGaWx0ZXJzIjp0cnVlLCJkc1JlcXVpcmVkRmlsdGVycyI6W10sImRhdGFzZXQiOnsiZGF0YXNldFR5cGUiOjEsImRhdGFzZXRJZCI6Il9fVklaX0RBVEFTT1VSQ0VfXyJ9LCJyb3ciOjEwMCwiZGltZW5zaW9ucyI6eyJsYWJlbGVkQ29uY2VwdHMiOlt7ImtleSI6InByaW1hcnkiLCJ2YWx1ZSI6eyJjb25jZXB0TmFtZXMiOlsicXRfdXM3Znc3N2QxZCJdfX1dfSwibWV0cmljcyI6eyJsYWJlbGVkQ29uY2VwdHMiOlt7ImtleSI6InByaW1hcnkiLCJ2YWx1ZSI6eyJjb25jZXB0TmFtZXMiOlsicXRfd3M3Znc3N2QxZCJdfX1dfSwiYmFyQ2hhcnRQcm9wZXJ0eSI6eyJvdmVycmlkZUF4aXNBdXRvIjp0cnVlLCJzZXJpZXNQcm9wZXJ0eSI6W3sic2VyaWVzQXhpc0luZGV4IjowfSx7InNlcmllc0F4aXNJbmRleCI6MX0seyJzZXJpZXNBeGlzSW5kZXgiOjB9LHsic2VyaWVzQXhpc0luZGV4IjowfSx7InNlcmllc0F4aXNJbmRleCI6MH0seyJzZXJpZXNBeGlzSW5kZXgiOjB9LHsic2VyaWVzQXhpc0luZGV4IjowfSx7InNlcmllc0F4aXNJbmRleCI6MH0seyJzZXJpZXNBeGlzSW5kZXgiOjB9LHsic2VyaWVzQXhpc0luZGV4IjowfSx7InNlcmllc0F4aXNJbmRleCI6MH0seyJzZXJpZXNBeGlzSW5kZXgiOjB9LHsic2VyaWVzQXhpc0luZGV4IjowfSx7InNlcmllc0F4aXNJbmRleCI6MH0seyJzZXJpZXNBeGlzSW5kZXgiOjB9LHsic2VyaWVzQXhpc0luZGV4IjowfSx7InNlcmllc0F4aXNJbmRleCI6MH0seyJzZXJpZXNBeGlzSW5kZXgiOjB9LHsic2VyaWVzQXhpc0luZGV4IjowfSx7InNlcmllc0F4aXNJbmRleCI6MH1dLCJyZWZlcmVuY2VMaW5lUHJvcGVydHkiOltdLCJyZWZlcmVuY2VCYW5kUHJvcGVydHkiOltdLCJiYWNrZ3JvdW5kQW5kQm9yZGVyUHJvcGVydHkiOnsiYm9yZGVyIjp7Im9wYWNpdHkiOjAsInNpemUiOjAsInJhZGl1cyI6MH19LCJpbnRlcnZhbFByb3BlcnR5IjpbXX0sImNvbXBvbmVudFByb3BlcnR5TWlncmF0aW9uU3RhdHVzIjoyfX0sImNvbmNlcHREZWZzIjpbeyJpZCI6InQwLnF0X3dzN2Z3NzdkMWQiLCJuYW1lIjoicXRfd3M3Znc3N2QxZCIsIm5hbWVzcGFjZSI6InQwIiwicXVlcnlUaW1lVHJhbnNmb3JtYXRpb24iOnsiZGF0YVRyYW5zZm9ybWF0aW9uIjp7InNvdXJjZUZpZWxkTmFtZSI6InBhdGllbnRfY291bnQiLCJhZ2dyZWdhdGlvbiI6Nn19fSx7ImlkIjoidDAucXRfdXM3Znc3N2QxZCIsIm5hbWUiOiJxdF91czdmdzc3ZDFkIiwibmFtZXNwYWNlIjoidDAiLCJxdWVyeVRpbWVUcmFuc2Zvcm1hdGlvbiI6eyJkYXRhVHJhbnNmb3JtYXRpb24iOnsic291cmNlRmllbGROYW1lIjoicHJvamVjdF9zaG9ydF9uYW1lIn19fV0sImF0dHJpYnV0ZUNvbmZpZyI6eyJjb21wb25lbnRBdHRyaWJ1dGUiOnsiZGF0YXNvdXJjZUNvbmZpZ1ZlcnNpb24iOjIsImRpc3BsYXlDb25maWdWZXJzaW9uIjowLCJoZWlnaHQiOjU3MSwid2lkdGgiOjE3NzEsInRvcCI6MCwibGVmdCI6MH19LCJjb21wb25lbnRJZCI6Il9fVklaX0NIQVJUX0lEX18iLCJ0eXBlIjoic2ltcGxlLWJhcmNoYXJ0IiwicHJlc2V0IjoiaG9yaXpvbnRhbC1zdGFja2VkIiwiYmVoYXZpb3IiOnsibWFwVmFsdWUiOnsiZW50cnkiOlt7ImtleSI6Im9uU29ydCIsInZhbHVlIjp7ImFycmF5VmFsdWUiOnsidmFsdWUiOlt7Im1hcFZhbHVlIjp7ImVudHJ5IjpbeyJrZXkiOiJhY3Rpb24iLCJ2YWx1ZSI6eyJzdHJWYWx1ZSI6InNvcnQifX0seyJrZXkiOiJpc0NvbnRyb2wiLCJ2YWx1ZSI6eyJib29sVmFsdWUiOmZhbHNlfX0seyJrZXkiOiJpbml0IiwidmFsdWUiOnsibWFwVmFsdWUiOnsiZW50cnkiOlt7ImtleSI6InNvcnRPcHRpb25zIiwidmFsdWUiOnsibWFwVmFsdWUiOnsiZW50cnkiOlt7ImtleSI6InNvcnREYXRhIiwidmFsdWUiOnsiYXJyYXlWYWx1ZSI6eyJ2YWx1ZSI6W3sibWFwVmFsdWUiOnsiZW50cnkiOlt7ImtleSI6InNvcnREaXIiLCJ2YWx1ZSI6eyJpbnQ2NFZhbHVlIjoiMSJ9fSx7ImtleSI6InNvcnRDb2x1bW4iLCJ2YWx1ZSI6eyJtYXBWYWx1ZSI6eyJlbnRyeSI6W3sia2V5IjoibmFtZSIsInZhbHVlIjp7InN0clZhbHVlIjoicXRfd3M3Znc3N2QxZCJ9fSx7ImtleSI6ImRhdGFzZXROcyIsInZhbHVlIjp7InN0clZhbHVlIjoiZDAifX0seyJrZXkiOiJ0YWJsZU5zIiwidmFsdWUiOnsic3RyVmFsdWUiOiJ0MCJ9fSx7ImtleSI6ImRhdGFUcmFuc2Zvcm1hdGlvbiIsInZhbHVlIjp7Im1hcFZhbHVlIjp7ImVudHJ5IjpbeyJrZXkiOiJzb3VyY2VGaWVsZE5hbWUiLCJ2YWx1ZSI6eyJzdHJWYWx1ZSI6InBhdGllbnRfY291bnQifX0seyJrZXkiOiJhZ2dyZWdhdGlvbiIsInZhbHVlIjp7ImludDY0VmFsdWUiOiI2In19XX19fV19fX1dfX1dfX19LHsia2V5IjoiaXNOZXdTb3J0Q29uZmlnIiwidmFsdWUiOnsiYm9vbFZhbHVlIjp0cnVlfX1dfX19XX19fV19fV19fX0seyJrZXkiOiJvblByZVNvcnQiLCJ2YWx1ZSI6eyJhcnJheVZhbHVlIjp7InZhbHVlIjpbeyJtYXBWYWx1ZSI6eyJlbnRyeSI6W3sia2V5IjoiYWN0aW9uIiwidmFsdWUiOnsic3RyVmFsdWUiOiJwcmVzb3J0In19LHsia2V5IjoiaXNDb250cm9sIiwidmFsdWUiOnsiYm9vbFZhbHVlIjpmYWxzZX19LHsia2V5IjoiaW5pdCIsInZhbHVlIjp7Im1hcFZhbHVlIjp7ImVudHJ5IjpbeyJrZXkiOiJzb3J0T3B0aW9ucyIsInZhbHVlIjp7Im1hcFZhbHVlIjp7ImVudHJ5IjpbeyJrZXkiOiJzb3J0RGF0YSIsInZhbHVlIjp7ImFycmF5VmFsdWUiOnsidmFsdWUiOltdfX19LHsia2V5IjoiaXNOZXdTb3J0Q29uZmlnIiwidmFsdWUiOnsiYm9vbFZhbHVlIjp0cnVlfX1dfX19XX19fV19fV19fX1dfX19LCJmaWx0ZXJzIjpbXSwiY2hhcnRJbnRlcmFjdGlvbnMiOlt7ImJlaGF2aW9yVHlwZSI6Im9uU29ydCIsInNvcnRQYXJhbWV0ZXJWYWx1ZSI6eyJzb3J0T3B0aW9ucyI6eyJpc05ld1NvcnRDb25maWciOnRydWUsInNvcnREYXRhIjpbeyJzb3J0RGlyIjoxLCJzb3J0Q29sdW1uIjp7Im5hbWUiOiJxdF82OTZnMDg3ZDFkIiwiZGF0YXNldE5zIjoiZDAiLCJ0YWJsZU5zIjoidDAiLCJkYXRhVHJhbnNmb3JtYXRpb24iOnsic291cmNlRmllbGROYW1lIjoicHJvamVjdF9zaG9ydF9uYW1lIiwiYWdncmVnYXRpb24iOjN9fX1dfX19XSwidmVyc2lvbiI6MX0SCW91dHB1dF9kZhoWChJwcm9qZWN0X3Nob3J0X25hbWUQARoRCg1wYXRpZW50X2NvdW50EAI=

import google.colabsqlviz.explore_dataframe as _vizcell
_vizcell.explore_dataframe(df_or_df_name='seq_and_patient_count_by_cancer_type', uuid='CC1849E3-13FF-4391-9870-F8F0EA5D4D5B', config_str='CuAZeyJjaGFydENvbmZpZyI6eyJkYXRhc291cmNlSWQiOiJfX1ZJWl9EQVRBU09VUkNFX18iLCJwcm9wZXJ0eUNvbmZpZyI6eyJjb21wb25lbnRQcm9wZXJ0eSI6eyJzb3J0IjpbeyJzb3J0RGlyIjoxLCJzb3J0Q29sdW1uIjoicXRfd3M3Znc3N2QxZCJ9XSwiYnJlYWtkb3duQ29uZmlnIjpbXSwiZmlsdGVycyI6W10sImluaGVyaXRGaWx0ZXJzIjp0cnVlLCJkc1JlcXVpcmVkRmlsdGVycyI6W10sImRhdGFzZXQiOnsiZGF0YXNldFR5cGUiOjEsImRhdGFzZXRJZCI6Il9fVklaX0RBVEFTT1VSQ0VfXyJ9LCJyb3ciOjEwMCwiZGltZW5zaW9ucyI6eyJsYWJlbGVkQ29uY2VwdHMiOlt7ImtleSI6InByaW1hcnkiLCJ2YWx1ZSI6eyJjb25jZXB0TmFtZXMiOlsicXRfdXM3Znc3N2QxZCJdfX1dfSwibWV0cmljcyI6eyJsYWJlbGVkQ29uY2VwdHMiOlt7ImtleSI6InByaW1hcnkiLCJ2YWx1ZSI6eyJjb25jZXB0TmFtZXMiOlsicXRfd3M3Znc3N2QxZCJdfX1dfSwiYmFyQ2hhcnRQcm9wZXJ0eSI6eyJvdmVycmlkZUF4aXNBdXRvIjp0cnVlLCJzZXJpZXNQcm9wZXJ0eSI6W3sic2VyaWVzQXhpc0luZGV4IjowfSx7InNlcmllc0F4aXNJbmRleCI6MX0seyJzZXJpZXNBeGlzSW5kZXgiOjB9LHsic2VyaWVzQXhpc0luZGV4IjowfSx7InNlcmllc0F4aXNJbmRleCI6MH0seyJzZXJpZXNBeGlzSW5kZXgiOjB9LHsic2VyaWVzQXhpc0luZGV4IjowfSx7InNlcmllc0F4aXNJbmRleCI6MH0seyJzZXJpZXNBeGlzSW5kZXgiOjB9LHsic2VyaWVzQXhpc0luZGV4IjowfSx7InNlcmllc0F4aXNJbmRleCI6MH0seyJzZXJpZXNBeGlzSW5kZXgiOjB9LHsic2VyaWVzQXhpc0luZGV4IjowfSx7InNlcmllc0F4aXNJbmRleCI6MH0seyJzZXJpZXNBeGlzSW5kZXgiOjB9LHsic2VyaWVzQXhpc0luZGV4IjowfSx7InNlcmllc0F4aXNJbmRleCI6MH0seyJzZXJpZXNBeGlzSW5kZXgiOjB9LHsic2VyaWVzQXhpc0luZGV4IjowfSx7InNlcmllc0F4aXNJbmRleCI6MH1dLCJyZWZlcmVuY2VMaW5lUHJvcGVydHkiOltdLCJyZWZlcmVuY2VCYW5kUHJvcGVydHkiOltdLCJiYWNrZ3JvdW5kQW5kQm9yZGVyUHJvcGVydHkiOnsiYm9yZGVyIjp7Im9wYWNpdHkiOjAsInNpemUiOjAsInJhZGl1cyI6MH19LCJpbnRlcnZhbFByb3BlcnR5IjpbXX0sImNvbXBvbmVudFByb3BlcnR5TWlncmF0aW9uU3RhdHVzIjoyfX0sImNvbmNlcHREZWZzIjpbeyJpZCI6InQwLnF0X3dzN2Z3NzdkMWQiLCJuYW1lIjoicXRfd3M3Znc3N2QxZCIsIm5hbWVzcGFjZSI6InQwIiwicXVlcnlUaW1lVHJhbnNmb3JtYXRpb24iOnsiZGF0YVRyYW5zZm9ybWF0aW9uIjp7InNvdXJjZUZpZWxkTmFtZSI6InBhdGllbnRfY291bnQiLCJhZ2dyZWdhdGlvbiI6Nn19fSx7ImlkIjoidDAucXRfdXM3Znc3N2QxZCIsIm5hbWUiOiJxdF91czdmdzc3ZDFkIiwibmFtZXNwYWNlIjoidDAiLCJxdWVyeVRpbWVUcmFuc2Zvcm1hdGlvbiI6eyJkYXRhVHJhbnNmb3JtYXRpb24iOnsic291cmNlRmllbGROYW1lIjoicHJvamVjdF9zaG9ydF9uYW1lIn19fV0sImF0dHJpYnV0ZUNvbmZpZyI6eyJjb21wb25lbnRBdHRyaWJ1dGUiOnsiZGF0YXNvdXJjZUNvbmZpZ1ZlcnNpb24iOjIsImRpc3BsYXlDb25maWdWZXJzaW9uIjowLCJoZWlnaHQiOjU3MSwid2lkdGgiOjE3NzEsInRvcCI6MCwibGVmdCI6MH19LCJjb21wb25lbnRJZCI6Il9fVklaX0NIQVJUX0lEX18iLCJ0eXBlIjoic2ltcGxlLWJhcmNoYXJ0IiwicHJlc2V0IjoiaG9yaXpvbnRhbC1zdGFja2VkIiwiYmVoYXZpb3IiOnsibWFwVmFsdWUiOnsiZW50cnkiOlt7ImtleSI6Im9uU29ydCIsInZhbHVlIjp7ImFycmF5VmFsdWUiOnsidmFsdWUiOlt7Im1hcFZhbHVlIjp7ImVudHJ5IjpbeyJrZXkiOiJhY3Rpb24iLCJ2YWx1ZSI6eyJzdHJWYWx1ZSI6InNvcnQifX0seyJrZXkiOiJpc0NvbnRyb2wiLCJ2YWx1ZSI6eyJib29sVmFsdWUiOmZhbHNlfX0seyJrZXkiOiJpbml0IiwidmFsdWUiOnsibWFwVmFsdWUiOnsiZW50cnkiOlt7ImtleSI6InNvcnRPcHRpb25zIiwidmFsdWUiOnsibWFwVmFsdWUiOnsiZW50cnkiOlt7ImtleSI6InNvcnREYXRhIiwidmFsdWUiOnsiYXJyYXlWYWx1ZSI6eyJ2YWx1ZSI6W3sibWFwVmFsdWUiOnsiZW50cnkiOlt7ImtleSI6InNvcnREaXIiLCJ2YWx1ZSI6eyJpbnQ2NFZhbHVlIjoiMSJ9fSx7ImtleSI6InNvcnRDb2x1bW4iLCJ2YWx1ZSI6eyJtYXBWYWx1ZSI6eyJlbnRyeSI6W3sia2V5IjoibmFtZSIsInZhbHVlIjp7InN0clZhbHVlIjoicXRfd3M3Znc3N2QxZCJ9fSx7ImtleSI6ImRhdGFzZXROcyIsInZhbHVlIjp7InN0clZhbHVlIjoiZDAifX0seyJrZXkiOiJ0YWJsZU5zIiwidmFsdWUiOnsic3RyVmFsdWUiOiJ0MCJ9fSx7ImtleSI6ImRhdGFUcmFuc2Zvcm1hdGlvbiIsInZhbHVlIjp7Im1hcFZhbHVlIjp7ImVudHJ5IjpbeyJrZXkiOiJzb3VyY2VGaWVsZE5hbWUiLCJ2YWx1ZSI6eyJzdHJWYWx1ZSI6InBhdGllbnRfY291bnQifX0seyJrZXkiOiJhZ2dyZWdhdGlvbiIsInZhbHVlIjp7ImludDY0VmFsdWUiOiI2In19XX19fV19fX1dfX1dfX19LHsia2V5IjoiaXNOZXdTb3J0Q29uZmlnIiwidmFsdWUiOnsiYm9vbFZhbHVlIjp0cnVlfX1dfX19XX19fV19fV19fX0seyJrZXkiOiJvblByZVNvcnQiLCJ2YWx1ZSI6eyJhcnJheVZhbHVlIjp7InZhbHVlIjpbeyJtYXBWYWx1ZSI6eyJlbnRyeSI6W3sia2V5IjoiYWN0aW9uIiwidmFsdWUiOnsic3RyVmFsdWUiOiJwcmVzb3J0In19LHsia2V5IjoiaXNDb250cm9sIiwidmFsdWUiOnsiYm9vbFZhbHVlIjpmYWxzZX19LHsia2V5IjoiaW5pdCIsInZhbHVlIjp7Im1hcFZhbHVlIjp7ImVudHJ5IjpbeyJrZXkiOiJzb3J0T3B0aW9ucyIsInZhbHVlIjp7Im1hcFZhbHVlIjp7ImVudHJ5IjpbeyJrZXkiOiJzb3J0RGF0YSIsInZhbHVlIjp7ImFycmF5VmFsdWUiOnsidmFsdWUiOltdfX19LHsia2V5IjoiaXNOZXdTb3J0Q29uZmlnIiwidmFsdWUiOnsiYm9vbFZhbHVlIjp0cnVlfX1dfX19XX19fV19fV19fX1dfX19LCJmaWx0ZXJzIjpbXSwiY2hhcnRJbnRlcmFjdGlvbnMiOlt7ImJlaGF2aW9yVHlwZSI6Im9uU29ydCIsInNvcnRQYXJhbWV0ZXJWYWx1ZSI6eyJzb3J0T3B0aW9ucyI6eyJpc05ld1NvcnRDb25maWciOnRydWUsInNvcnREYXRhIjpbeyJzb3J0RGlyIjoxLCJzb3J0Q29sdW1uIjp7Im5hbWUiOiJxdF82OTZnMDg3ZDFkIiwiZGF0YXNldE5zIjoiZDAiLCJ0YWJsZU5zIjoidDAiLCJkYXRhVHJhbnNmb3JtYXRpb24iOnsic291cmNlRmllbGROYW1lIjoicHJvamVjdF9zaG9ydF9uYW1lIiwiYWdncmVnYXRpb24iOjN9fX1dfX19XSwidmVyc2lvbiI6MX0SCW91dHB1dF9kZhoWChJwcm9qZWN0X3Nob3J0X25hbWUQARoRCg1wYXRpZW50X2NvdW50EAI=')

<IPython.core.display.Javascript object>

In [10]:
seq_and_patient_count_by_cancer_type['avg'] = seq_and_patient_count_by_cancer_type['row_count'] / seq_and_patient_count_by_cancer_type['patient_count']
seq_and_patient_count_by_cancer_type

,project_short_name,row_count,patient_count,avg
0,TCGA-BRCA,74677384,1095,68198.524201
1,TCGA-UCEC,35731096,557,64149.184919
2,TCGA-KIRC,37247696,533,69883.106942
3,TCGA-HNSC,34335824,521,65903.692898
4,TCGA-LUAD,36398400,517,70403.094778
5,TCGA-LGG,32394576,516,62780.186047
6,TCGA-THCA,34699808,505,68712.491089
7,TCGA-LUSC,34093168,501,68050.235529
8,TCGA-PRAD,33607856,497,67621.440644
9,TCGA-SKCM,28694072,469,61181.390192


**Key difference: sparse vs. dense tables.** Mutation tables are *sparse* — only mutated genes get rows, so a patient with 100 mutations has 100 rows. Expression tables are *dense* — every gene gets a row for every sample. The ~60,000–70,000 rows per patient roughly matches the number of annotated genes in the human genome (protein-coding, lncRNA, pseudogenes, and other non-coding transcripts).

### What types of genes are measured?

RNA-seq quantifies far more than just protein-coding genes. This query breaks down the ~60k gene entries per patient by `gene_type` to see how many are protein-coding, lncRNA, pseudogenes, miRNA, etc. This matters when deciding which genes to include in downstream analyses — for most multi-omic work, we'll filter to protein-coding genes.

In [11]:
# sql_engine: bigquery
# output_variable: count_by_gene_type
# start _sql
_sql = """
SELECT gene_type, COUNT(DISTINCT gene_name) AS gene_count
FROM `isb-cgc-bq.TCGA.RNAseq_hg38_gdc_current`
GROUP BY gene_type
ORDER BY gene_count DESC
""" # end _sql
from google.colab.sql import bigquery as _bqsqlcell
count_by_gene_type = _bqsqlcell.run(_sql)
count_by_gene_type

,gene_type,gene_count
0,protein_coding,19938
1,lncRNA,16882
2,processed_pseudogene,10160
3,unprocessed_pseudogene,2612
4,miRNA,1879
5,snRNA,1837
6,misc_RNA,1279
7,TEC,1057
8,transcribed_unprocessed_pseudogene,938
9,snoRNA,793


In [12]:
# dataframe: count_by_gene_type
# uuid: ADE7A356-1F7D-4DFF-A86D-97EBCD184216
# output_variable:
# config_str: Cp0QeyJjaGFydENvbmZpZyI6eyJkYXRhc291cmNlSWQiOiJfX1ZJWl9EQVRBU09VUkNFX18iLCJwcm9wZXJ0eUNvbmZpZyI6eyJjb21wb25lbnRQcm9wZXJ0eSI6eyJzb3J0IjpbeyJzb3J0RGlyIjoxLCJzb3J0Q29sdW1uIjoicXRfd3NyZmU2N2QxZCJ9XSwiYnJlYWtkb3duQ29uZmlnIjpbXSwiZmlsdGVycyI6W10sImluaGVyaXRGaWx0ZXJzIjp0cnVlLCJkc1JlcXVpcmVkRmlsdGVycyI6W10sImRhdGFzZXQiOnsiZGF0YXNldFR5cGUiOjEsImRhdGFzZXRJZCI6Il9fVklaX0RBVEFTT1VSQ0VfXyJ9LCJyb3ciOjEwLCJkaW1lbnNpb25zIjp7ImxhYmVsZWRDb25jZXB0cyI6W3sia2V5IjoicHJpbWFyeSIsInZhbHVlIjp7ImNvbmNlcHROYW1lcyI6WyJxdF92c3JmZTY3ZDFkIl19fV19LCJtZXRyaWNzIjp7ImxhYmVsZWRDb25jZXB0cyI6W3sia2V5IjoicHJpbWFyeSIsInZhbHVlIjp7ImNvbmNlcHROYW1lcyI6WyJxdF93c3JmZTY3ZDFkIl19fV19LCJwaWVDaGFydFByb3BlcnR5Ijp7InNsaWNlQ29sb3IiOltdLCJiYWNrZ3JvdW5kQW5kQm9yZGVyUHJvcGVydHkiOnsiYm9yZGVyIjp7Im9wYWNpdHkiOjAsInNpemUiOjAsInJhZGl1cyI6MH19fSwiY29tcG9uZW50UHJvcGVydHlNaWdyYXRpb25TdGF0dXMiOjJ9fSwiY29uY2VwdERlZnMiOlt7ImlkIjoidDAucXRfdnNyZmU2N2QxZCIsIm5hbWUiOiJxdF92c3JmZTY3ZDFkIiwibmFtZXNwYWNlIjoidDAiLCJxdWVyeVRpbWVUcmFuc2Zvcm1hdGlvbiI6eyJkYXRhVHJhbnNmb3JtYXRpb24iOnsic291cmNlRmllbGROYW1lIjoiZ2VuZV90eXBlIn19fSx7ImlkIjoidDAucXRfd3NyZmU2N2QxZCIsIm5hbWUiOiJxdF93c3JmZTY3ZDFkIiwibmFtZXNwYWNlIjoidDAiLCJxdWVyeVRpbWVUcmFuc2Zvcm1hdGlvbiI6eyJkYXRhVHJhbnNmb3JtYXRpb24iOnsic291cmNlRmllbGROYW1lIjoiZ2VuZV9jb3VudCIsImFnZ3JlZ2F0aW9uIjo2fX19XSwiYXR0cmlidXRlQ29uZmlnIjp7ImNvbXBvbmVudEF0dHJpYnV0ZSI6eyJkYXRhc291cmNlQ29uZmlnVmVyc2lvbiI6MiwiZGlzcGxheUNvbmZpZ1ZlcnNpb24iOjAsImhlaWdodCI6NTcxLCJ3aWR0aCI6MTc3MSwidG9wIjowLCJsZWZ0IjowfX0sImNvbXBvbmVudElkIjoiX19WSVpfQ0hBUlRfSURfXyIsInR5cGUiOiJzaW1wbGUtcGllY2hhcnQiLCJwcmVzZXQiOiJkZWZhdWx0IiwiYmVoYXZpb3IiOnsibWFwVmFsdWUiOnsiZW50cnkiOlt7ImtleSI6Im9uU29ydCIsInZhbHVlIjp7ImFycmF5VmFsdWUiOnsidmFsdWUiOlt7Im1hcFZhbHVlIjp7ImVudHJ5IjpbeyJrZXkiOiJhY3Rpb24iLCJ2YWx1ZSI6eyJzdHJWYWx1ZSI6InNvcnQifX0seyJrZXkiOiJpc0NvbnRyb2wiLCJ2YWx1ZSI6eyJib29sVmFsdWUiOmZhbHNlfX0seyJrZXkiOiJpbml0IiwidmFsdWUiOnsibWFwVmFsdWUiOnsiZW50cnkiOlt7ImtleSI6InNvcnRPcHRpb25zIiwidmFsdWUiOnsibWFwVmFsdWUiOnsiZW50cnkiOlt7ImtleSI6InNvcnREYXRhIiwidmFsdWUiOnsiYXJyYXlWYWx1ZSI6eyJ2YWx1ZSI6W3sibWFwVmFsdWUiOnsiZW50cnkiOlt7ImtleSI6InNvcnREaXIiLCJ2YWx1ZSI6eyJpbnQ2NFZhbHVlIjoiMSJ9fSx7ImtleSI6InNvcnRDb2x1bW4iLCJ2YWx1ZSI6eyJtYXBWYWx1ZSI6eyJlbnRyeSI6W3sia2V5IjoibmFtZSIsInZhbHVlIjp7InN0clZhbHVlIjoicXRfd3NyZmU2N2QxZCJ9fSx7ImtleSI6ImRhdGFzZXROcyIsInZhbHVlIjp7InN0clZhbHVlIjoiZDAifX0seyJrZXkiOiJ0YWJsZU5zIiwidmFsdWUiOnsic3RyVmFsdWUiOiJ0MCJ9fSx7ImtleSI6ImRhdGFUcmFuc2Zvcm1hdGlvbiIsInZhbHVlIjp7Im1hcFZhbHVlIjp7ImVudHJ5IjpbeyJrZXkiOiJzb3VyY2VGaWVsZE5hbWUiLCJ2YWx1ZSI6eyJzdHJWYWx1ZSI6ImdlbmVfY291bnQifX0seyJrZXkiOiJhZ2dyZWdhdGlvbiIsInZhbHVlIjp7ImludDY0VmFsdWUiOiI2In19XX19fV19fX1dfX1dfX19LHsia2V5IjoiaXNOZXdTb3J0Q29uZmlnIiwidmFsdWUiOnsiYm9vbFZhbHVlIjp0cnVlfX1dfX19XX19fV19fV19fX1dfX19LCJmaWx0ZXJzIjpbXSwiY2hhcnRJbnRlcmFjdGlvbnMiOltdLCJ2ZXJzaW9uIjoxfRoOCgpnZW5lX2NvdW50EAIaDQoJZ2VuZV90eXBlEAE=

import google.colabsqlviz.explore_dataframe as _vizcell
_vizcell.explore_dataframe(df_or_df_name='count_by_gene_type', uuid='ADE7A356-1F7D-4DFF-A86D-97EBCD184216', config_str='Cp0QeyJjaGFydENvbmZpZyI6eyJkYXRhc291cmNlSWQiOiJfX1ZJWl9EQVRBU09VUkNFX18iLCJwcm9wZXJ0eUNvbmZpZyI6eyJjb21wb25lbnRQcm9wZXJ0eSI6eyJzb3J0IjpbeyJzb3J0RGlyIjoxLCJzb3J0Q29sdW1uIjoicXRfd3NyZmU2N2QxZCJ9XSwiYnJlYWtkb3duQ29uZmlnIjpbXSwiZmlsdGVycyI6W10sImluaGVyaXRGaWx0ZXJzIjp0cnVlLCJkc1JlcXVpcmVkRmlsdGVycyI6W10sImRhdGFzZXQiOnsiZGF0YXNldFR5cGUiOjEsImRhdGFzZXRJZCI6Il9fVklaX0RBVEFTT1VSQ0VfXyJ9LCJyb3ciOjEwLCJkaW1lbnNpb25zIjp7ImxhYmVsZWRDb25jZXB0cyI6W3sia2V5IjoicHJpbWFyeSIsInZhbHVlIjp7ImNvbmNlcHROYW1lcyI6WyJxdF92c3JmZTY3ZDFkIl19fV19LCJtZXRyaWNzIjp7ImxhYmVsZWRDb25jZXB0cyI6W3sia2V5IjoicHJpbWFyeSIsInZhbHVlIjp7ImNvbmNlcHROYW1lcyI6WyJxdF93c3JmZTY3ZDFkIl19fV19LCJwaWVDaGFydFByb3BlcnR5Ijp7InNsaWNlQ29sb3IiOltdLCJiYWNrZ3JvdW5kQW5kQm9yZGVyUHJvcGVydHkiOnsiYm9yZGVyIjp7Im9wYWNpdHkiOjAsInNpemUiOjAsInJhZGl1cyI6MH19fSwiY29tcG9uZW50UHJvcGVydHlNaWdyYXRpb25TdGF0dXMiOjJ9fSwiY29uY2VwdERlZnMiOlt7ImlkIjoidDAucXRfdnNyZmU2N2QxZCIsIm5hbWUiOiJxdF92c3JmZTY3ZDFkIiwibmFtZXNwYWNlIjoidDAiLCJxdWVyeVRpbWVUcmFuc2Zvcm1hdGlvbiI6eyJkYXRhVHJhbnNmb3JtYXRpb24iOnsic291cmNlRmllbGROYW1lIjoiZ2VuZV90eXBlIn19fSx7ImlkIjoidDAucXRfd3NyZmU2N2QxZCIsIm5hbWUiOiJxdF93c3JmZTY3ZDFkIiwibmFtZXNwYWNlIjoidDAiLCJxdWVyeVRpbWVUcmFuc2Zvcm1hdGlvbiI6eyJkYXRhVHJhbnNmb3JtYXRpb24iOnsic291cmNlRmllbGROYW1lIjoiZ2VuZV9jb3VudCIsImFnZ3JlZ2F0aW9uIjo2fX19XSwiYXR0cmlidXRlQ29uZmlnIjp7ImNvbXBvbmVudEF0dHJpYnV0ZSI6eyJkYXRhc291cmNlQ29uZmlnVmVyc2lvbiI6MiwiZGlzcGxheUNvbmZpZ1ZlcnNpb24iOjAsImhlaWdodCI6NTcxLCJ3aWR0aCI6MTc3MSwidG9wIjowLCJsZWZ0IjowfX0sImNvbXBvbmVudElkIjoiX19WSVpfQ0hBUlRfSURfXyIsInR5cGUiOiJzaW1wbGUtcGllY2hhcnQiLCJwcmVzZXQiOiJkZWZhdWx0IiwiYmVoYXZpb3IiOnsibWFwVmFsdWUiOnsiZW50cnkiOlt7ImtleSI6Im9uU29ydCIsInZhbHVlIjp7ImFycmF5VmFsdWUiOnsidmFsdWUiOlt7Im1hcFZhbHVlIjp7ImVudHJ5IjpbeyJrZXkiOiJhY3Rpb24iLCJ2YWx1ZSI6eyJzdHJWYWx1ZSI6InNvcnQifX0seyJrZXkiOiJpc0NvbnRyb2wiLCJ2YWx1ZSI6eyJib29sVmFsdWUiOmZhbHNlfX0seyJrZXkiOiJpbml0IiwidmFsdWUiOnsibWFwVmFsdWUiOnsiZW50cnkiOlt7ImtleSI6InNvcnRPcHRpb25zIiwidmFsdWUiOnsibWFwVmFsdWUiOnsiZW50cnkiOlt7ImtleSI6InNvcnREYXRhIiwidmFsdWUiOnsiYXJyYXlWYWx1ZSI6eyJ2YWx1ZSI6W3sibWFwVmFsdWUiOnsiZW50cnkiOlt7ImtleSI6InNvcnREaXIiLCJ2YWx1ZSI6eyJpbnQ2NFZhbHVlIjoiMSJ9fSx7ImtleSI6InNvcnRDb2x1bW4iLCJ2YWx1ZSI6eyJtYXBWYWx1ZSI6eyJlbnRyeSI6W3sia2V5IjoibmFtZSIsInZhbHVlIjp7InN0clZhbHVlIjoicXRfd3NyZmU2N2QxZCJ9fSx7ImtleSI6ImRhdGFzZXROcyIsInZhbHVlIjp7InN0clZhbHVlIjoiZDAifX0seyJrZXkiOiJ0YWJsZU5zIiwidmFsdWUiOnsic3RyVmFsdWUiOiJ0MCJ9fSx7ImtleSI6ImRhdGFUcmFuc2Zvcm1hdGlvbiIsInZhbHVlIjp7Im1hcFZhbHVlIjp7ImVudHJ5IjpbeyJrZXkiOiJzb3VyY2VGaWVsZE5hbWUiLCJ2YWx1ZSI6eyJzdHJWYWx1ZSI6ImdlbmVfY291bnQifX0seyJrZXkiOiJhZ2dyZWdhdGlvbiIsInZhbHVlIjp7ImludDY0VmFsdWUiOiI2In19XX19fV19fX1dfX1dfX19LHsia2V5IjoiaXNOZXdTb3J0Q29uZmlnIiwidmFsdWUiOnsiYm9vbFZhbHVlIjp0cnVlfX1dfX19XX19fV19fV19fX1dfX19LCJmaWx0ZXJzIjpbXSwiY2hhcnRJbnRlcmFjdGlvbnMiOltdLCJ2ZXJzaW9uIjoxfRoOCgpnZW5lX2NvdW50EAIaDQoJZ2VuZV90eXBlEAE=')

<IPython.core.display.Javascript object>

## Part 2: CPTAC — Adding proteomics to the picture

TCGA provides genomics (mutations) and transcriptomics (RNA-seq), but not proteomics. The [Clinical Proteomic Tumor Analysis Consortium (CPTAC)](https://proteomics.cancer.gov/programs/cptac) fills that gap with mass-spectrometry-based protein and phosphoprotein quantification for many of the same cancer types. ISB-CGC hosts CPTAC data in BigQuery alongside TCGA, making multi-omic joins possible.

Let's explore what's available.

In [13]:
# sql_engine: bigquery
# output_variable: CPTAC_table_names
# start _sql
_sql = """
SELECT table_name
FROM `isb-cgc-bq.CPTAC.INFORMATION_SCHEMA.TABLES`
""" # end _sql
from google.colab.sql import bigquery as _bqsqlcell
CPTAC_table_names = _bqsqlcell.run(_sql)
CPTAC_table_names

,table_name
0,quant_phosphoproteome_CPTAC_LSCC_discovery_stu...
1,clinical_CPTAC3_other_pdc_current
2,quant_proteome_CPTAC_GBM_discovery_study_pdc_c...
3,quant_phosphoproteome_PTRC_HGSOC_FFPE_validati...
4,quant_proteome_beat_AML_baseline_clinical_pdc_...
5,quant_phosphoproteome_PTRC_HGSOC_FFPE_discover...
6,quant_phosphoproteome_CPTAC_GBM_confirmatory_s...
7,RNAseq_hg38_gdc_current
8,quant_proteome_prospective_ovarian_JHU_pdc_cur...
9,quant_phosphoproteome_prospective_colon_PNNL_l...


In [14]:
CPTAC_table_names.to_string()

'                                                                                       table_name\n0                                    quant_phosphoproteome_CPTAC_LSCC_discovery_study_pdc_current\n1                                                               clinical_CPTAC3_other_pdc_current\n2                                            quant_proteome_CPTAC_GBM_discovery_study_pdc_current\n3                                    quant_phosphoproteome_PTRC_HGSOC_FFPE_validation_pdc_current\n4                                           quant_proteome_beat_AML_baseline_clinical_pdc_current\n5                                     quant_phosphoproteome_PTRC_HGSOC_FFPE_discovery_pdc_current\n6                                  quant_phosphoproteome_CPTAC_GBM_confirmatory_study_pdc_current\n7                                                                         RNAseq_hg38_gdc_current\n8                                              quant_proteome_prospective_ovarian_JHU_pdc_current\n9        

### CPTAC clinical table schema

Before querying CPTAC clinical data, we need to know what columns are available — particularly patient identifiers and disease annotations that we'll use to join with the proteomics and genomics tables.

In [15]:
# sql_engine: bigquery
# output_variable: clinical_gdc_columns
# start _sql
_sql = """
SELECT column_name
FROM `isb-cgc-bq.CPTAC.INFORMATION_SCHEMA.COLUMNS`
WHERE table_name = 'clinical_gdc_current'
ORDER BY ordinal_position
""" # end _sql
from google.colab.sql import bigquery as _bqsqlcell
clinical_gdc_columns = _bqsqlcell.run(_sql)
clinical_gdc_columns

,column_name
0,submitter_id
1,case_id
2,follow__count
3,diag__treat__count
4,primary_site
5,disease_type
6,index_date
7,consent_type
8,days_to_consent
9,lost_to_followup


### How many patients per CPTAC project?

A quick count of distinct `case_id` values per project tells us the overall scale of each CPTAC phase.

In [16]:
# sql_engine: bigquery
# output_variable: CPTAC_case_count_by_project
# start _sql
_sql = """
    SELECT proj__name, proj__project_id, COUNT(DISTINCT case_id) AS patient_count
    FROM `isb-cgc-bq.CPTAC.clinical_gdc_current`
    GROUP BY proj__name, proj__project_id
    ORDER BY patient_count DESC
""" # end _sql
from google.colab.sql import bigquery as _bqsqlcell
CPTAC_case_count_by_project = _bqsqlcell.run(_sql)
CPTAC_case_count_by_project

,proj__name,proj__project_id,patient_count
0,"CPTAC-Brain, Head and Neck, Kidney, Lung, Panc...",CPTAC-3,1683
1,"CPTAC-Breast, Colon, Ovary",CPTAC-2,342


### CPTAC case counts by cancer type

CPTAC-2 and CPTAC-3 are umbrella projects that each cover several cancer types. Breaking down by `primary_site` and `disease_type` shows exactly which cancers are represented and at what sample sizes — important for knowing which cross-omic comparisons are statistically viable.

In [17]:
# sql_engine: bigquery
# output_variable: CPTAC_case_count_by_cancer_type
# start _sql
_sql = """
SELECT proj__project_id, primary_site, disease_type, COUNT(DISTINCT case_id) AS patient_count
FROM `isb-cgc-bq.CPTAC.clinical_gdc_current`
GROUP BY proj__project_id, primary_site, disease_type
ORDER BY proj__project_id, patient_count DESC
""" # end _sql
from google.colab.sql import bigquery as _bqsqlcell
CPTAC_case_count_by_cancer_type = _bqsqlcell.run(_sql)
CPTAC_case_count_by_cancer_type

,proj__project_id,primary_site,disease_type,patient_count
0,CPTAC-2,Breast,Ductal and Lobular Neoplasms,115
1,CPTAC-2,Colon,Adenomas and Adenocarcinomas,105
2,CPTAC-2,Ovary,"Cystic, Mucinous and Serous Neoplasms",71
3,CPTAC-2,Other and unspecified female genital organs,"Cystic, Mucinous and Serous Neoplasms",21
4,CPTAC-2,Breast,<NA>,16
5,CPTAC-2,Retroperitoneum and peritoneum,"Cystic, Mucinous and Serous Neoplasms",10
6,CPTAC-2,Breast,"Cystic, Mucinous and Serous Neoplasms",1
7,CPTAC-2,Breast,Squamous Cell Neoplasms,1
8,CPTAC-2,Rectum,Adenomas and Adenocarcinomas,1
9,CPTAC-2,Breast,Adenomas and Adenocarcinomas,1


### Do CPTAC-2 and CPTAC-3 tables share identifiers?

Different CPTAC phases used different proteomics pipelines. Before we can join mutation, expression, and protein data across phases, we need to check whether CPTAC-2 (prospective) and CPTAC-3 (discovery) tables share the same identifier columns (`case_id`, `sample_id`, `aliquot_id`, etc.).

In [19]:
# sql_engine: bigquery
# output_variable: CPTAC2_columns
# start _sql
_sql = """
-- What identifiers does a CPTAC-2 prospective table use?
SELECT column_name
FROM `isb-cgc-bq.CPTAC.INFORMATION_SCHEMA.COLUMNS`
WHERE table_name = 'quant_proteome_prospective_breast_BI_pdc_current'
ORDER BY ordinal_position
""" # end _sql
from google.colab.sql import bigquery as _bqsqlcell
CPTAC2_columns = _bqsqlcell.run(_sql)
CPTAC2_columns

,column_name
0,case_id
1,sample_id
2,aliquot_id
3,aliquot_submitter_id
4,aliquot_run_metadata_id
5,study_name
6,protein_abundance_log2ratio
7,gene_id
8,gene_symbol
9,NCBI_gene_id


In [20]:
# sql_engine: bigquery
# output_variable: CPTAC3_columns
# start _sql
_sql = """
-- What identifiers does a CPTAC-3 discovery table use?
SELECT column_name
FROM `isb-cgc-bq.CPTAC.INFORMATION_SCHEMA.COLUMNS`
WHERE table_name = 'quant_proteome_CPTAC_UCEC_discovery_study_pdc_current'
ORDER BY ordinal_position
""" # end _sql
from google.colab.sql import bigquery as _bqsqlcell
CPTAC3_columns = _bqsqlcell.run(_sql)
CPTAC3_columns

,column_name
0,case_id
1,sample_id
2,aliquot_id
3,aliquot_submitter_id
4,aliquot_run_metadata_id
5,study_name
6,protein_abundance_log2ratio
7,gene_id
8,gene_symbol
9,NCBI_gene_id
